In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath("../.."))
from quantile_forest import RandomForestQuantileRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from tinyconformal.regressor import ConformalizedQuantileRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np
from tinyconformal.utils import MultiQuantileRegressor

In [ ]:
EXAMPLES_DIR = Path.cwd().parent
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))
from utils.plot_utils import plot_prediction_intervals

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_calib, y_train, y_calib = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:
coverage = 0.90
alpha = 1 - coverage
quantiles = [alpha/2, 1 - alpha/2]

In [ ]:
rf = RandomForestQuantileRegressor(random_state=42, default_quantiles=quantiles, n_jobs=-1, oob_score=True, max_depth=int(np.ceil(np.log2(len(X_train)) - 1)))
rf.fit(X_train, y_train)

In [ ]:
rf.predict(X_test)

In [ ]:
reg = ConformalizedQuantileRegressor(
    rf,
    alpha=alpha,
)
reg.fit(X_train, y_train, oob=True)

In [ ]:
reg

In [ ]:
y_pred_intervals = reg.predict_interval(X_test, alpha=0.2)
y_pred = reg.predict(X_test, alpha=0.2)

In [ ]:
reg.evaluate(X_test, y_test, alpha=0.2)

In [ ]:
plot_prediction_intervals(y_pred_intervals[:10], y_pred[:10], y_test[:10], fig_type="png")

# MultiQuantileRegressor - HistGradientBoostingRegressor

In [ ]:
mq_model = MultiQuantileRegressor(
        base_estimator=HistGradientBoostingRegressor(loss="quantile", random_state=42),
        quantiles=(0.1, 0.50, 0.90),
    )
mq_model.fit(X_train, y_train)
cqr = ConformalizedQuantileRegressor(learner=mq_model, alpha=0.2)
cqr.fit(X_calib, y_calib)

In [ ]:
cqr.predict_interval(X_test)

In [ ]:
cqr.evaluate(X_test, y_test)

In [ ]:
bounds = cqr.predict_interval(X_test)
y_pred = cqr.predict(X_test)

In [ ]:
plot_prediction_intervals(bounds[:10], y_pred[:10], y_test[:10], fig_type="png")